In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


BASE_DIR = Path.cwd()

MODEL_PATH = BASE_DIR / "models" / "isolation_forest.joblib"
FEATURES_PATH = BASE_DIR / "models" / "features.joblib"

THRESHOLD = 0.02


def load_model():
    """Load the trained model and its required feature list."""
    model = joblib.load(MODEL_PATH)
    features = joblib.load(FEATURES_PATH)
    return model, features


def predict_csv(file):
    """Validate a network-flow CSV and flag anomalous flows."""

    model, features = load_model()

    # Read uploaded CSV
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()

    # Validate required columns
    missing = [col for col in features if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep only the features used during training
    X = df[features].copy()

    # Convert values to numeric and handle invalid values
    for col in features:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    X.replace([np.inf, -np.inf], np.nan, inplace=True)

    # The model's pipeline handles missing values using its imputer
    scores = model.decision_function(X)

    # Lower scores indicate more anomalous traffic
    anomaly = scores < THRESHOLD

    # Return results for the backend or frontend
    results = df.copy()
    results["anomaly_score"] = scores
    results["prediction"] = np.where(
        anomaly, "Anomaly", "Normal"
    )

    return results

In [8]:
import os

print("Current directory:", os.getcwd())
print("Files in data:", os.listdir("data"))

Current directory: /Users/himanshu/Documents/my documents/hackathon/cyvanta/ml
Files in data: ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip']


In [9]:
import zipfile

with zipfile.ZipFile("data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip") as z:
    z.extractall("data")

print("Dataset extracted!")

Dataset extracted!


In [10]:
from pipeline import predict_csv

result = predict_csv("data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

print(result["prediction"].value_counts())
result.head()

prediction
Normal     115497
Anomaly    110248
Name: count, dtype: int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,anomaly_score,prediction
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.127690,Normal
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.127690,Normal


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = result["Label"].str.strip().map({
    "BENIGN": "Normal",
    "DDoS": "Anomaly"
})

print(confusion_matrix(y_true, result["prediction"]))
print(classification_report(y_true, result["prediction"]))

[[84776 43251]
 [25472 72246]]
              precision    recall  f1-score   support

     Anomaly       0.77      0.66      0.71    128027
      Normal       0.63      0.74      0.68     97718

    accuracy                           0.70    225745
   macro avg       0.70      0.70      0.69    225745
weighted avg       0.71      0.70      0.70    225745

